In [1]:
import pandas as pd
import uuid
from pathlib import Path
import json

In [2]:
RAW_DATA_DIR = Path("../data/01_raw")

In [12]:
df_processed = pd.read_csv(RAW_DATA_DIR/"isco_clean_taxonomy.csv")
df_processed.head()



,id,code,level,label,definition,examples,parentCode,isLeaf
0,7cba0c3a,1,1,Managers,"Managers plan, direct, coordinate and evaluate...",Occupations in this major group are classified...,NaN,0
1,9667e9d1,11,2,"Chief Executives, Senior Officials and Legisla...","Chief executives, senior officials and legisla...",Occupations in this sub-major group are classi...,1.0,0
2,64236362,111,3,Legislators and Senior Officials,"Legislators and senior officials determine, fo...",Occupations in this minor group are classified...,11.0,0
3,900276e9,1111,4,Legislators,"Legislators determine, formulate, and direct p...",Examples of the occupations classified here:\n...,111.0,1
4,d66c8e86,1112,4,Senior Government Officials,Senior government officials advise governments...,Examples of the occupations classified here:\n...,111.0,1


In [13]:
df_processed['taxonomyKey'] = 'ISCO'

In [14]:
df_processed.rename(columns={'definition': 'text'}, inplace=True)

In [15]:
df_processed.columns

Index(['id', 'code', 'level', 'label', 'text', 'examples', 'parentCode',
       'isLeaf', 'taxonomyKey'],
      dtype='object')

In [8]:

df_processed.to_csv(RAW_DATA_DIR/"labeled_training_data.csv", index=False)

In [ ]:
# Build training-style JSON payload from the taxonomy leaves

def format_code(value):
    if pd.isna(value):
        return None
    if isinstance(value, float) and value.is_integer():
        value_str = str(int(value))
    else:
        value_str = str(value)
    return value_str[:-2] if value_str.endswith('.0') else value_str


df_processed["code"] = df_processed["code"].apply(format_code)
df_processed["parentCode"] = df_processed["parentCode"].apply(format_code)
df_processed["level"] = df_processed["level"].astype(int)
df_processed["taxonomyKey"] = df_processed["taxonomyKey"].astype(str)

records = df_processed.to_dict(orient="records")
node_lookup = {(row["code"], int(row["level"])): row for row in records}


def build_annotations(start_row):
    annotations = []
    current_key = (start_row["code"], int(start_row["level"]))
    visited = set()

    while current_key and current_key not in visited:
        node = node_lookup.get(current_key)
        if not node:
            break

        level = int(node["level"])
        annotations.append({"level": level, "nodeCode": node["code"]})
        visited.add(current_key)

        parent_code = node.get("parentCode")
        if not parent_code:
            break

        current_key = (parent_code, level - 1)

    return sorted(annotations, key=lambda ann: ann["level"])


sentences = []
leaf_nodes = df_processed[df_processed["isLeaf"] == 1]
for _, row in leaf_nodes.iterrows():
    row_data = row.to_dict()
    description = "" if pd.isna(row_data["text"]) else str(row_data["text"]).strip()
    examples = "" if pd.isna(row_data["examples"]) else str(row_data["examples"]).strip()
    if examples:
        job_description = f"{description} Examples: {examples}" if description else examples
    else:
        job_description = description

    annotations = build_annotations(row_data)
    if not annotations:
        continue

    sentences.append(
        {
            "sentenceId": str(row_data["id"] or uuid.uuid4()),
            "fields": {
                "job_title": str(row_data["label"]).strip(),
                "job_description": job_description,
            },
            "annotations": annotations,
        }
    )

training_payload = {
    "taxonomyKey": df_processed["taxonomyKey"].iloc[0],
    "sentences": sentences,
}

output_path = RAW_DATA_DIR / "isco_taxonomy_sentences.json"
with open(output_path, "w") as f:
    json.dump(training_payload, f, indent=2)

output_path
